# Notebook 1: Data Pipeline

Pulls commodity futures price data from Yahoo Finance for the crack spread strategy.

**Tickers used:**
- `CL=F` — WTI Crude Oil futures ($/bbl)
- `RB=F` — RBOB Gasoline futures ($/gallon)
- `HO=F` — Heating Oil futures ($/gallon)

**Output:** `../data/raw_prices.csv`

---

### Note on column handling

`yfinance` returns columns in **alphabetical order by ticker** (`CL=F`, `HO=F`,
`RB=F`), not in the order requested. Assigning names positionally with
`raw.columns = list(tickers.keys())` therefore mislabels RBOB and HeatOil,
swapping the two product series. Columns are selected explicitly by ticker
below, with assertions on price levels so the error cannot recur silently.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('../data', exist_ok=True)

In [ ]:
# ============================================================
# DOWNLOAD
# 15 years of daily data. The longer sample is what makes the
# out-of-sample trade count large enough to draw conclusions from.
# ============================================================

tickers = {
    'WTI':     'CL=F',   # WTI crude,      $/bbl
    'RBOB':    'RB=F',   # RBOB gasoline,  $/gallon
    'HeatOil': 'HO=F'    # Heating oil,    $/gallon
}

START_DATE = '2010-01-01'
END_DATE   = '2025-01-01'

dl = yf.download(
    list(tickers.values()),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True
)['Close']

print('Columns as returned by yfinance:', list(dl.columns))
print('(alphabetical, NOT the order requested)')

# Select each column by ticker name so ordering cannot silently swap series
raw = pd.DataFrame({name: dl[ticker] for name, ticker in tickers.items()})
raw = raw.dropna()

print(f'\nDownloaded {len(raw)} trading days')
print(f'Date range: {raw.index[0].date()} to {raw.index[-1].date()}')
raw.head()

In [ ]:
# ============================================================
# SANITY CHECK - catches mislabelled columns immediately
# ============================================================

print('Typical levels over the period:')
print(f"  WTI      ${raw['WTI'].mean():7.2f}/bbl    (expect roughly $50-80)")
print(f"  RBOB     ${raw['RBOB'].mean():7.3f}/gal   (expect roughly $1.7-2.3)")
print(f"  HeatOil  ${raw['HeatOil'].mean():7.3f}/gal   (expect roughly $2.0-2.6)")
print()

spread_check = (raw['HeatOil'] - raw['RBOB']).mean()
print(f"HeatOil - RBOB = ${spread_check:.3f}/gal on average")
print('Heating oil normally trades slightly ABOVE RBOB over a long sample.')
if spread_check < -0.10:
    print('*** WARNING: strongly negative - columns may be swapped. ***')
else:
    print('Looks consistent with correctly labelled series.')

# Crude in $/bbl should be far larger than products quoted in $/gal
assert raw['WTI'].mean() > 20, 'WTI mean implausibly low - check column mapping'
assert raw['RBOB'].mean() < 10, 'RBOB mean implausibly high - check column mapping'
assert raw['HeatOil'].mean() < 10, 'HeatOil mean implausibly high - check column mapping'
print('\nColumn mapping assertions passed.')

In [ ]:
# Basic data quality checks
print('Missing values:')
print(raw.isnull().sum())
print()
print('Summary statistics:')
print(raw.describe().round(3))

In [ ]:
# Plot raw prices
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(raw.index, raw['WTI'], color='#2c3e50', linewidth=1)
axes[0].set_ylabel('WTI Crude ($/bbl)')
axes[0].set_title(f'Commodity Futures Prices - Raw Data ({START_DATE[:4]}-{END_DATE[:4]})')
axes[0].grid(alpha=0.3)

axes[1].plot(raw.index, raw['RBOB'], color='#e74c3c', linewidth=1)
axes[1].set_ylabel('RBOB Gasoline ($/gallon)')
axes[1].grid(alpha=0.3)

axes[2].plot(raw.index, raw['HeatOil'], color='#3498db', linewidth=1)
axes[2].set_ylabel('Heating Oil ($/gallon)')
axes[2].set_xlabel('Date')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/raw_prices_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save to CSV
raw.to_csv('../data/raw_prices.csv')
print(f'Saved {len(raw)} rows to ../data/raw_prices.csv')